# 1. Azure Network Security

This notebook covers the Azure network security services that appear on the SC-900 exam. We'll simulate how each one works and explain when to use it.

## The services at a glance

| Service | What it protects | Layer |
|---------|-----------------|-------|
| **Azure DDoS Protection** | Public IPs from volumetric attacks | Network (L3/L4) |
| **Azure Firewall** | VNets — centralized network traffic filtering | Network (L3-L7) |
| **Web Application Firewall (WAF)** | Web apps from OWASP attacks (SQLi, XSS) | Application (L7) |
| **Network Security Groups (NSGs)** | Subnets and NICs — port/IP filtering | Network (L3/L4) |
| **Azure Virtual Network (VNet)** | Network isolation and segmentation | Network |
| **Azure Bastion** | VMs — secure RDP/SSH without public IPs | Access |
| **Azure Key Vault** | Secrets, keys, certificates | Data |

---
## 1. Virtual Networks (VNets) and Subnets

A **VNet** is your private network in Azure. Resources inside a VNet can talk to each other by default. Resources in *different* VNets are isolated unless you explicitly connect them (peering, VPN, etc.).

**Subnets** divide a VNet into segments. Each subnet can have its own NSG (firewall rules).

```
┌── VNet: 10.0.0.0/16 ──────────────────────────────────┐
│                                                         │
│  ┌── Subnet: web (10.0.1.0/24) ──┐   ┌── Subnet: db (10.0.2.0/24) ──┐  │
│  │  🌐 Web Server VM              │   │  🗄️ Database VM               │  │
│  │  NSG: allow 80,443 from any    │   │  NSG: allow 5432 from web     │  │
│  │       allow 22 from Bastion    │   │       deny all else           │  │
│  └────────────────────────────────┘   └────────────────────────────────┘  │
│                                                         │
│  ┌── Subnet: AzureBastionSubnet ──┐                     │
│  │  🔒 Azure Bastion              │                     │
│  └────────────────────────────────┘                     │
└─────────────────────────────────────────────────────────┘
```

### Exam tip
- VNets provide **isolation** (different VNets can't communicate by default).
- Subnets provide **segmentation** within a VNet.
- Both are **free** in Azure.

---
## 2. Network Security Groups (NSGs)

NSGs are **packet filters** — they allow or deny traffic based on source/destination IP, port, and protocol. Applied to subnets or individual NICs.

Each NSG has a list of **rules** evaluated by priority (lowest number = highest priority).

In [ ]:
import json

# Simulate an NSG with rules
NSG_RULES = [
    {'priority': 100, 'name': 'AllowHTTPS',   'direction': 'inbound', 'src': '*',           'dst_port': 443,  'action': 'allow'},
    {'priority': 110, 'name': 'AllowHTTP',    'direction': 'inbound', 'src': '*',           'dst_port': 80,   'action': 'allow'},
    {'priority': 200, 'name': 'AllowSSHFromBastion', 'direction': 'inbound', 'src': '10.0.3.0/24', 'dst_port': 22, 'action': 'allow'},
    {'priority': 300, 'name': 'DenyRDP',      'direction': 'inbound', 'src': '*',           'dst_port': 3389, 'action': 'deny'},
    {'priority': 65000, 'name': 'AllowVNetInbound', 'direction': 'inbound', 'src': 'VirtualNetwork', 'dst_port': '*', 'action': 'allow'},
    {'priority': 65500, 'name': 'DenyAllInbound',  'direction': 'inbound', 'src': '*',           'dst_port': '*', 'action': 'deny'},
]

def evaluate_nsg(src_ip: str, dst_port: int, rules: list) -> dict:
    """Evaluate NSG rules in priority order (lowest number first)."""
    sorted_rules = sorted(rules, key=lambda r: r['priority'])
    for rule in sorted_rules:
        port_match = rule['dst_port'] == '*' or rule['dst_port'] == dst_port
        src_match = rule['src'] == '*' or rule['src'] == 'VirtualNetwork' or src_ip.startswith(rule['src'].split('/')[0][:6])
        if port_match and src_match:
            return {
                'matched_rule': rule['name'],
                'priority': rule['priority'],
                'action': '✅ ALLOW' if rule['action'] == 'allow' else '🚫 DENY',
            }
    return {'matched_rule': 'implicit-deny', 'action': '🚫 DENY'}

print('NSG Rules (sorted by priority):')
for r in sorted(NSG_RULES, key=lambda r: r['priority']):
    print(f'  {r["priority"]:>5}  {r["action"]:>5}  {r["name"]:<25} src={r["src"]:<20} port={r["dst_port"]}')

print('\n=== Traffic evaluation ===')
tests = [
    ('Internet user → HTTPS (443)',  '203.0.113.1', 443),
    ('Internet user → SSH (22)',     '203.0.113.1', 22),
    ('Bastion subnet → SSH (22)',    '10.0.3.5',    22),
    ('Internet user → RDP (3389)',   '203.0.113.1', 3389),
    ('VNet peer → PostgreSQL (5432)','10.0.2.10',   5432),
]
for desc, src, port in tests:
    result = evaluate_nsg(src, port, NSG_RULES)
    print(f'  {desc:<40} → {result["action"]}  (rule: {result["matched_rule"]})')

### Key NSG facts for the exam

- Rules evaluated by **priority** (100-4096 for custom rules).
- Default rules: AllowVNetInbound, AllowAzureLoadBalancerInbound, DenyAllInbound.
- Applied at **subnet** or **NIC** level (both are evaluated — most restrictive wins).
- NSGs are **stateful** — if inbound is allowed, the response is automatically allowed.
- **Free** to use (no cost per NSG).

---
## 3. Azure Firewall vs NSG vs WAF

These three overlap but serve different purposes:

| Feature | NSG | Azure Firewall | WAF |
|---------|-----|---------------|------|
| **Layer** | L3/L4 (IP, port) | L3-L7 (includes FQDN, TLS) | L7 (HTTP only) |
| **Scope** | Subnet/NIC | Entire VNet (centralized) | Web apps (App Gateway / Front Door) |
| **FQDN filtering** | No | Yes (e.g., allow *.microsoft.com) | N/A |
| **TLS inspection** | No | Yes (Premium) | Yes |
| **OWASP protection** | No | No | Yes (SQLi, XSS, etc.) |
| **Threat intelligence** | No | Yes (block known malicious IPs) | Yes |
| **Cost** | Free | ~$900/month+ | Included with App Gateway |
| **Use case** | Micro-segmentation | Central egress/ingress control | Protect web APIs |

### Exam tip
- **NSG** = subnet-level packet filter, free, simple.
- **Azure Firewall** = centralized, enterprise-grade, FQDN filtering, threat intel.
- **WAF** = specifically for HTTP attacks (OWASP Top 10).

---
## 4. Azure DDoS Protection

Two tiers:

| | DDoS Network Protection (Basic) | DDoS Network Protection (Standard) |
|-|------------|----------|
| **Cost** | Free (on by default) | ~$2,944/month |
| **Scope** | All Azure services | Per-VNet opt-in |
| **Features** | Automatic traffic monitoring | Adaptive tuning, attack analytics, rapid response team, cost protection |

**Exam tip**: Basic protection is **always on** for all Azure resources. Standard adds monitoring, alerting, and the DDoS Rapid Response (DRR) team.

---
## 5. Azure Bastion

**Problem**: to SSH/RDP into a VM, you used to need a public IP on the VM — a huge security risk.

**Bastion solution**: a managed PaaS service that provides secure RDP/SSH directly in the Azure portal over TLS. The VM never needs a public IP.

```
Admin → Azure Portal (HTTPS) → Bastion → VM (private IP only)
```

- No public IP needed on VMs.
- No NSG rules for RDP (3389) or SSH (22) from the internet.
- Requires a dedicated subnet named `AzureBastionSubnet` (/26 or larger).

---
## 6. Azure Key Vault

Centralized store for three types of sensitive data:

| Type | Examples | Operations |
|------|---------|-------------|
| **Secrets** | API keys, connection strings, passwords | Get, Set, List, Delete |
| **Keys** | Encryption keys (RSA, EC) | Encrypt, Decrypt, Sign, Verify, Wrap, Unwrap |
| **Certificates** | TLS/SSL certificates | Create, Import, Renew |

### Key Vault access control

- **RBAC** (recommended): use Azure roles like `Key Vault Secrets User`.
- **Access policies** (legacy): per-principal permissions.
- **Network restrictions**: private endpoints, firewall rules.

### Exam tip
- Key Vault supports **soft delete** (recover accidentally deleted items) and **purge protection** (even admins can't permanently delete during the retention period).
- HSM-backed keys use FIPS 140-2 Level 2 (standard) or Level 3 (managed HSM).

In [ ]:
# Simulate Key Vault operations
class MockKeyVault:
    def __init__(self):
        self._secrets = {}
        self._deleted = {}
    
    def set_secret(self, name: str, value: str):
        self._secrets[name] = value
        print(f'  ✅ Secret "{name}" set (value hidden)')
    
    def get_secret(self, name: str, role: str) -> str:
        allowed_roles = {'Key Vault Secrets User', 'Key Vault Administrator', 'Owner'}
        if role not in allowed_roles:
            print(f'  ❌ Access denied — role "{role}" cannot read secrets')
            return None
        val = self._secrets.get(name)
        if val:
            print(f'  ✅ Secret "{name}" retrieved')
        else:
            print(f'  ❌ Secret "{name}" not found')
        return val
    
    def delete_secret(self, name: str):
        if name in self._secrets:
            self._deleted[name] = self._secrets.pop(name)
            print(f'  🗑️ Secret "{name}" soft-deleted (recoverable for 90 days)')
        else:
            print(f'  ❌ Secret "{name}" not found')
    
    def recover_secret(self, name: str):
        if name in self._deleted:
            self._secrets[name] = self._deleted.pop(name)
            print(f'  ♻️ Secret "{name}" recovered from soft-delete!')
        else:
            print(f'  ❌ Secret "{name}" not in deleted items')

vault = MockKeyVault()
print('=== Key Vault operations ===')
vault.set_secret('db-connection-string', 'Server=prod.db;Password=s3cret')
vault.set_secret('api-key', 'sk-abc123xyz')
print()

print('--- Read as Key Vault Secrets User ---')
vault.get_secret('db-connection-string', 'Key Vault Secrets User')
print()

print('--- Read as Reader (insufficient) ---')
vault.get_secret('db-connection-string', 'Reader')
print()

print('--- Soft delete + recover ---')
vault.delete_secret('api-key')
vault.get_secret('api-key', 'Key Vault Secrets User')  # not found
vault.recover_secret('api-key')
vault.get_secret('api-key', 'Key Vault Secrets User')  # recovered!

---
## Summary

| Service | Purpose | Key exam fact |
|---------|---------|---------------|
| **VNet** | Network isolation | Different VNets are isolated by default |
| **NSG** | Subnet/NIC packet filter | Rules by priority, stateful, free |
| **Azure Firewall** | Centralized L3-L7 filtering | FQDN filtering, threat intel |
| **WAF** | Protect web apps from OWASP | SQLi, XSS protection |
| **DDoS Protection** | Absorb volumetric attacks | Basic is free and always on |
| **Bastion** | Secure RDP/SSH without public IPs | Requires AzureBastionSubnet |
| **Key Vault** | Secrets, keys, certificates | Soft delete + purge protection |

**Next**: [Notebook 2 — Key Vault and Defender for Cloud](02_key_vault_and_defender.ipynb)